# コラム: 不毛な台地

Barren plateau（BP）とは、VQEやQCLなどの変分量子回路を用いたアルゴリズムにしばしば発生する問題であり、これらのアルゴリズムで不可欠なパラメータ最適化を困難にするものである。
正確な主張は数学的に複雑なのでここでは省略するが、簡単に言うと BP とは

> 十分な表現能力を持った変分量子状態 $\ket{\psi(\boldsymbol{\theta})} = U(\boldsymbol{\theta})\ket{0\cdots0}$ について、物理量 $O$ に対する期待値 $E_O(\boldsymbol{\theta}) = \ev{O}{\psi(\boldsymbol{\theta})}$ を考える。
> このとき、パラメータ $\boldsymbol{\theta}$ をランダムにとると、ほぼ確率 1 で勾配 $\pdv{E_O(\boldsymbol{\theta})}{\boldsymbol{\theta}}$ の全成分が 0 になる（量子ビット数 $n$ について指数関数的に小さくなる）。
> よって、パラメータ最適化を進めることが困難になる。

という現象のことである。これを簡単な例で数値計算してみよう。

まず、$n$ 量子ビットの変分量子回路としては、1量子ビット回転ゲートと全量子ビット間の CNOT ゲートを交互に $d=n$ 層繰り返したものを用いる。
物理量 $O$ は 1 番目の量子ビットに対するパウリ $Z$ 演算子 $O=Z_1$ とする。

```python
import numpy as np
import matplotlib.pyplot as plt

from quri_parts.circuit import QuantumCircuit
from quri_parts.circuit import UnboundParametricQuantumCircuit
from quri_parts.core.state import quantum_state, apply_circuit
from quri_parts.core.operator import Operator, pauli_label
from quri_parts.qulacs.estimator import create_qulacs_vector_estimator

def create_PQC(n_qubits, c_depth):
    circuit = UnboundParametricQuantumCircuit(n_qubits)
    for _ in range(c_depth):
        for i in range(n_qubits):
            circuit.add_ParametricRX_gate(i)
            circuit.add_ParametricRZ_gate(i)
            circuit.add_ParametricRX_gate(i)
            for j in range(i + 1, n_qubits):
                circuit.add_CNOT_gate(i, j)
    return circuit

rng = np.random.default_rng(2025)
obs = Operator({pauli_label("Z0"): 1})
estimator = create_qulacs_vector_estimator()
n_trials = 100
```

ランダムなパラメータ $\boldsymbol{\theta}$ を $n_\mathrm{trials}$ 回生成して、期待値 $\ev{Z_1}{\psi(\boldsymbol{\theta})}$ を計算し、その平均と標準偏差を記録する。
本来は期待値そのものではなくその勾配を計算するべきなのだが、一般的には期待値の標準偏差が 0 に近づけば勾配も 0 に近づくので、今回は期待値を計算する。

```python
n_qubits_list = np.arange(4, 14)
ave_list = []
std_list = []
for n_qubits in n_qubits_list:
    c_depth = n_qubits
    circuit = create_PQC(n_qubits, c_depth)
    exps = []
    for _ in range(n_trials):
        theta = 2 * np.pi * rng.random(circuit.parameter_count)
        state = quantum_state(n_qubits)
        state = apply_circuit(circuit.bind_parameters(theta), state)
        exp = estimator(obs, state).value.real
        exps.append(exp)
    ave_list.append(np.abs(np.mean(exps)))
    std_list.append(np.std(exps))

plt.semilogy(n_qubits_list, std_list, "o", label="Std.")
plt.xlabel("n_qubits")
plt.legend()
plt.show()
```

[図 5.13 プレースホルダ: 期待値標準偏差の減衰]

プロットを見ると、期待値 $\ev{Z_1}{\psi(\boldsymbol{\theta})}$ の標準偏差が量子ビット数 $n$ について指数関数的に減衰していることが読み取れる。
つまり $n$ が大きい場合、どのようなランダムパラメータ $\boldsymbol{\theta}$ を持ってきても、期待値はほぼ一定の値をとってしまい、その勾配もほぼ 0 になる。
ゆえに期待値の最小化・最大化が不可能になり、VQEやQCLの実行が困難になる。

BPを避けるためには、パラメータの一部のみをランダムに初期化して最適化を開始する、最適解に近い良い初期値からスタートする、などの方法があり、現在も研究が進められている。